# 03 · The same strategy in a popular library

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kolyamkl/qm-fintech-quant-workshop/blob/main/notebooks/03_compare_with_library.ipynb)

*QM FinTech Society quant workshop · Led by **Adnan Salehin** (Lead SWE & VP, JPMorgan)*

You've built strategies on our small engine. Real projects often use a library. Here's the same 50/200 crossover in [backtesting.py](https://kernc.github.io/backtesting.py/) — and why the numbers don't match exactly.

> **Running in Google Colab?** If Colab warns that this notebook wasn't written by Google, click **Run anyway**. Then use **File → Save a copy in Drive** so your changes are kept.

In [1]:
# ▶ Setup — run this cell first.
# In Google Colab it downloads the workshop repo; on your laptop it just finds the code.
import os, subprocess, sys

REPO = "qm-fintech-quant-workshop"
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", f"https://github.com/kolyamkl/{REPO}.git"], check=True)
    os.chdir(REPO)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # the code lives one folder up from notebooks/
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Re-load your edited .py files automatically (handy when you change strategies/template.py)
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.precision", 4)

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "backtesting"], check=True)

In [2]:
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Jupyter Notebook detected")
    from backtesting import Backtest, Strategy
    from backtesting.lib import crossover
    from backtesting.test import SMA

from backtester import load_prices, run_backtest
from strategies import ma_crossover

spy = load_prices("SPY")


class SmaCross(Strategy):
    fast = 50
    slow = 200

    def init(self):
        self.fast_ma = self.I(SMA, self.data.Close, self.fast)
        self.slow_ma = self.I(SMA, self.data.Close, self.slow)

    def next(self):
        if crossover(self.fast_ma, self.slow_ma):    # fast crosses above slow
            self.buy()
        elif crossover(self.slow_ma, self.fast_ma):  # fast crosses below slow
            self.position.close()


# 0.0005 = 5 bps; finalize_trades closes the last open trade at the end, like our engine does
bt = Backtest(spy, SmaCross, cash=10_000, commission=0.0005, finalize_trades=True)
stats = bt.run()
stats

Start                     2016-01-04 00:00:00
End                       2026-10-09 00:00:00
Duration                   3931 days 00:00:00
Exposure Time [%]                    57.86558
Equity Final [$]                  21401.46961
Equity Peak [$]                   21488.89885
Commissions [$]                      53.30129
Return [%]                           114.0147
Buy & Hold Return [%]               329.26787
Return (Ann.) [%]                     7.34001
Volatility (Ann.) [%]                 14.5922
CAGR [%]                              7.32558
Sharpe Ratio                          0.50301
Sortino Ratio                         0.75499
Calmar Ratio                          0.22075
Alpha [%]                           -82.06784
Beta                                  0.59551
Max. Drawdown [%]                   -33.24965
Avg. Drawdown [%]                    -1.89083
Max. Drawdown Duration      543 days 00:00:00
Avg. Drawdown Duration       26 days 00:00:00
# Trades                          

## Side by side

In [3]:
ours = run_backtest(spy, ma_crossover(spy, fast=50, slow=200), cost_bps=5)

comparison = pd.DataFrame(
    {
        "Total return (%)": [ours.metrics["Total return"] * 100, stats["Return [%]"]],
        "Buy & hold return (%)": [ours.benchmark_metrics["Total return"] * 100, stats["Buy & Hold Return [%]"]],
        "Max drawdown (%)": [ours.metrics["Max drawdown"] * 100, stats["Max. Drawdown [%]"]],
        "Number of trades": [int(ours.metrics["Number of trades"]), int(stats["# Trades"])],
    },
    index=["Our engine", "backtesting.py"],
)
comparison.round(2)

,Total return (%),Buy & hold return (%),Max drawdown (%),Number of trades
Our engine,191.84,360.55,-33.72,5
backtesting.py,114.01,329.27,-33.25,4


## Why the numbers differ

Both are "right" — they just make different simplifying assumptions:

| | Our engine | backtesting.py |
|---|---|---|
| **When you trade** | at the close of the signal day | at the **next day's open** |
| **How much you buy** | exactly 100% of the portfolio (fractional) | **whole shares** only, leaving some cash idle |
| **When the first trade happens** | as soon as the 200-day average exists and fast > slow | only on an actual **crossover** event, so if fast is already above slow when the warm-up ends, it waits for the next one |
| **Buy & hold measured from** | the first day of data | the first bar after the indicator warm-up |
| **How costs are charged** | % of portfolio per position change | commission % on each order's value |

Lesson: **always know your backtester's assumptions.** Two honest tools can give different numbers for the "same" strategy — and a dishonest assumption (like the look-ahead bug in notebook 01) can make any strategy look great.

**✏️ Try it:** `bt.plot()` opens an interactive chart in a new tab/output (it may not display on GitHub).